<a href="https://colab.research.google.com/github/vignesh-potharaj/gen-ai/blob/main/SustainabilityManager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required packages
!pip install -q -U google-genai pandas tabulate

import os
import pandas as pd
from google.colab import userdata
from google.genai import types, client

# Retrieve Gemini API Key from Colab Secrets
api_key = userdata.get('GEMINI_API_KEY')

# Initialize Gemini Client
ai = client.Client(api_key=api_key)

print("Environment setup complete! Gemini Client initialized for AI Sustainability Manager.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.6/258.6 kB 12.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.2 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
Environment setup complete! Gemini Client initialized for AI Sustainability Manager.


In [2]:
# Create simulated IoT household energy dataset (Perception Data)
raw_sensor_data = {
    "device_id": ["DEV_001", "DEV_002", "DEV_003", "DEV_004", "DEV_005", "DEV_006"],
    "device_name": ["HVAC Air Conditioner", "LED Living Room Lights", "EV Home Charger", "Refrigerator", "Desktop Workstation", "Old Space Heater"],
    "power_draw_kw": [3.5, 0.04, 7.2, 0.15, 0.45, 1.8],
    "daily_usage_hours": [8.0, 5.0, 4.0, 24.0, 10.0, 6.0],
    "is_peak_hours": [True, False, True, True, True, True]
}

# Load into Pandas DataFrame to represent agent's perceptual state
df_perception = pd.DataFrame(raw_sensor_data)

# Calculate Daily Consumption (kWh) = Power Draw (kW) * Daily Hours
df_perception["daily_kwh"] = df_perception["power_draw_kw"] * df_perception["daily_usage_hours"]

print("=== AGENT PERCEPTION LAYER: INGESTED IOT SENSOR DATA ===")
print(df_perception.to_string(index=False))

=== AGENT PERCEPTION LAYER: INGESTED IOT SENSOR DATA ===
device_id            device_name  power_draw_kw  daily_usage_hours  is_peak_hours  daily_kwh
  DEV_001   HVAC Air Conditioner           3.50                8.0           True       28.0
  DEV_002 LED Living Room Lights           0.04                5.0          False        0.2
  DEV_003        EV Home Charger           7.20                4.0           True       28.8
  DEV_004           Refrigerator           0.15               24.0           True        3.6
  DEV_005    Desktop Workstation           0.45               10.0           True        4.5
  DEV_006       Old Space Heater           1.80                6.0           True       10.8


In [3]:
# Function to determine sustainability category (Decision Layer)
def classify_energy_impact(row):
    # High impact: High kWh or high power draw during peak hours
    if row["daily_kwh"] > 10.0 or (row["power_draw_kw"] >= 2.0 and row["is_peak_hours"]):
        return "High Carbon Footprint / Heavy Consumer"
    elif row["daily_kwh"] >= 2.0:
        return "Moderate Consumer"
    else:
        return "Eco Efficient / Low Impact"

# Function to generate automated eco recommendations (Action Layer)
def generate_eco_recommendation(row):
    if "EV Home Charger" in row["device_name"]:
        return "Action: Schedule charging during off-peak hours (11 PM - 6 AM) to reduce grid strain and electricity cost."
    elif "Air Conditioner" in row["device_name"]:
        return "Action: Increase thermostat setpoint by 2°C and enable smart eco-mode during peak tariff hours."
    elif "Space Heater" in row["device_name"]:
        return "Action: Replace inefficient resistive heater with a smart heat pump or set automated timer."
    elif "Desktop Workstation" in row["device_name"]:
        return "Action: Enable automatic sleep mode after 15 minutes of inactivity."
    elif "Lights" in row["device_name"]:
        return "Action: System optimal. Maintain current schedule."
    else:
        return "Action: Baseline load steady. Monitor standby draw."

# Apply Decision logic
df_perception["sustainability_category"] = df_perception.apply(classify_energy_impact, axis=1)

# Apply Action logic
df_perception["recommended_action"] = df_perception.apply(generate_eco_recommendation, axis=1)

print("=== AGENT DECISION & ACTION LAYER: CATEGORIZATION & RECOMMENDATIONS ===")
for idx, row in df_perception.iterrows():
    print(f"Device: {row['device_name']}")
    print(f"  - Daily Consumption: {row['daily_kwh']:.2f} kWh (Peak Hours: {row['is_peak_hours']})")
    print(f"  - Category: {row['sustainability_category']}")
    print(f"  - {row['recommended_action']}\n")

=== AGENT DECISION & ACTION LAYER: CATEGORIZATION & RECOMMENDATIONS ===
Device: HVAC Air Conditioner
  - Daily Consumption: 28.00 kWh (Peak Hours: True)
  - Category: High Carbon Footprint / Heavy Consumer
  - Action: Increase thermostat setpoint by 2°C and enable smart eco-mode during peak tariff hours.

Device: LED Living Room Lights
  - Daily Consumption: 0.20 kWh (Peak Hours: False)
  - Category: Eco Efficient / Low Impact
  - Action: System optimal. Maintain current schedule.

Device: EV Home Charger
  - Daily Consumption: 28.80 kWh (Peak Hours: True)
  - Category: High Carbon Footprint / Heavy Consumer
  - Action: Schedule charging during off-peak hours (11 PM - 6 AM) to reduce grid strain and electricity cost.

Device: Refrigerator
  - Daily Consumption: 3.60 kWh (Peak Hours: True)
  - Category: Moderate Consumer
  - Action: Baseline load steady. Monitor standby draw.

Device: Desktop Workstation
  - Daily Consumption: 4.50 kWh (Peak Hours: True)
  - Category: Moderate Consumer


In [7]:
# Cell 4: Agent Architectural Analysis (Gemini Interpretation)

# 1. Define prompt asking Gemini to compare agent architectures in a sustainability context
agent_analysis_prompt = """
You are an expert AI Systems Architect and Sustainability Engineer.

Explain how 4 different classic AI Agent architectures would process and respond to the household IoT energy dataset (containing HVAC units, EV chargers, and lighting):

1. **Simple Reflex Agent**: How would it act based strictly on immediate Condition-Action rules?
2. **Goal-Based Agent**: How would it plan actions when given an explicit goal (e.g., "Keep total household daily energy below 15 kWh")?
3. **Utility-Based Agent**: How would it balance trade-offs using a mathematical utility function (e.g., balancing electricity costs vs. occupant comfort)?
4. **Learning Agent**: How would it continuously improve its performance over time using feedback loops, historical weather data, and occupant behavioral patterns?

Provide a clear, structured explanation with concrete examples for each agent type applied to this smart home scenario.
"""

# 2. Get list of available models for your API key
all_models = [m.name.replace("models/", "") for m in ai.models.list()]

# Find the first available Flash model (fallback to 'gemini-2.0-flash' or 'gemini-2.5-flash')
flash_candidates = [m for m in all_models if "flash" in m and "preview" not in m]
if flash_candidates:
    model_id = flash_candidates[0]
else:
    model_id = "gemini-2.0-flash"

print(f"Selected active model: {model_id}\n")

config = types.GenerateContentConfig(
    temperature=0.2,
    max_output_tokens=1500
)

# 3. Call generate_content with the clean model name string
response = ai.models.generate_content(
    model=model_id,
    contents=agent_analysis_prompt,
    config=config
)

print("=== GEMINI AGENT ARCHITECTURE ANALYSIS ===")
print(response.text)

Selected active model: gemini-2.5-flash



ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}